# 프로젝트 요약 — 최신 결과 기준

공공기관 AI·IT 사업 제안요청서(RFP)의 요구사항 조항을 **제안 실무자 시선**으로 셋으로 나눈다.
`통상수용`(그냥 받아들임) / `견적반영`(원가에 넣음) / `계약·질의검토`(계약 전에 따져봄).

이 노트북은 지금까지의 흐름을 한 자리에서 다시 계산한다. 숫자는 전부 `reports/current/`와
`data/labels/`에서 읽고, 여기서 새로 학습하는 것은 없다. 어느 셀도 `RFP_DATASET_VERSION`에
기대지 않는다(버전을 직접 지정한다).

## 흐름

| 단계 | 한 줄 | 자세한 노트북 |
|---|---|---|
| 1. 문제 | 조항 수백 개 중 실무자가 신경 쓰는 자리를 미리 골라주는 분류기 | [00](00_project_overview.ipynb) |
| 2. 데이터 | 나라장터 공고 13건의 hwp·xlsx에서 요구사항 표를 직접 파싱, 1,445건 | [01](01_dataset_pipeline.ipynb) [03](03_requirements_eda.ipynb) |
| 3. 라벨 | Claude에 판정 규칙 + 문서별 예시(앵커)를 주고 배치 라벨링. 조건을 바꿔 재실행해 흔들림 측정 | [02](02_labeling_experiment.ipynb) [04](04_anchor_pool_analysis.ipynb) [06](06_label_eda.ipynb) [17](17_rerun_agreement.ipynb) |
| 4. 모델 | 13문서 LODO 고정, 측정 하한 0.016을 미리 정함. TF-IDF → roberta 파인튜닝 → 다수결 앙상블 | [07](07_baseline_comparison.ipynb) [09](09_model_summary.ipynb) [15](15_finetuning.ipynb) [19](19_training_recipes.ipynb) [12](12_candidate_ensemble.ipynb) |
| 5. 한계 | 오답의 32%는 견적↔계약 경계이고, 그 자리는 텍스트가 아니라 문서 밖 사정이 정한다 | [13](13_label_boundary.ipynb) [20](20_boundary_cases.ipynb) [21](21_cluster_diagnostics.ipynb) [18](18_decision_structure.ipynb) |
| 6. 기각 | 마스킹·군집·트리·보조 헤드·맥락 재판정은 수치와 이유를 적고 닫음. 마지막으로 sLLM LoRA | [14](14_text_masking.ipynb) [08](08_embedding_comparison.ipynb) [10](10_explainable_classical_search.ipynb) [05](05_finetune_analysis.ipynb) |

결정 기록은 `docs/history/decisions-*.md`, 남은 일은 `docs/NEXT.md`.


In [ ]:
"""데이터 — 문서 13개, 조항 1,445건. 라벨셋은 v5(주)와 v7(규칙을 좁힌 것) 둘을 나란히 본다."""
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.labeling.label_dataset import load_label_dataset

LABELS = ("통상수용", "견적반영", "계약·질의검토")
rows = {v: load_label_dataset(version=v)[0] for v in ("v5", "v7")}
docs = sorted({r["document_id"] for r in rows["v5"]})
print(f"문서 {len(docs)}개, 조항 {len(rows['v5'])}건")

dist = pd.DataFrame({v: pd.Series([r["primary_action"] for r in rs]).value_counts() for v, rs in rows.items()}).loc[list(LABELS)]
dist_pct = (dist / dist.sum() * 100).round(1).astype(str) + "%"
print("\n라벨 분포"); print(pd.concat({"건수": dist, "비율": dist_pct}, axis=1).to_string())

per_doc = pd.crosstab(pd.Series([r["document_id"] for r in rows["v5"]], name="문서"),
                      pd.Series([r["primary_action"] for r in rows["v5"]], name="v5 라벨"))[list(LABELS)]
per_doc["합"] = per_doc.sum(axis=1)
print("\n문서별 (v5)"); print(per_doc.sort_values("합", ascending=False).to_string())
print(f"
→ 문서마다 크기가 {per_doc['합'].min()}~{per_doc['합'].max()}건으로 다르고 라벨 비율도 다르다. 그래서 무작위 분할이 아니라 문서 단위로 빼서 평가한다(LODO).")


In [ ]:
"""점수 — 같은 13문서 LODO에서 모델을 올린 순서. sLLM 줄은 결과가 기록돼 있으면 채워진다.

fold 평균 macro F1이 주 지표다. 측정 하한 0.016(2 SE) 아래의 차이는 "같다"로 읽는다.
"""
import matplotlib.pyplot as plt

def results(version):
    return json.loads((ROOT / "reports/current" / version / "finetune_results.json").read_text(encoding="utf-8"))

R = {v: results(v) for v in ("v5", "v7")}
ROWS = [("TF-IDF (word+char)", "singles", "word+char TF-IDF"),
        ("roberta-base s7", "singles", "ftB7"),
        ("roberta-large s42", "singles", "ftL42"),
        ("앙상블 wc+ftB7+ftL42", "ensembles", "wc+ftB7+ftL42"),
        ("sLLM Qwen2.5-7B LoRA", "singles", "llm42"),
        ("앙상블 + sLLM", "ensembles", "wc+ftB7+ftL42+llm42+e5")]

def cell(version, kind, key, field="fold_mean_macro_f1"):
    entry = R[version][kind].get(key)
    return entry[field] if entry else None

table = pd.DataFrame({v: {name: cell(v, kind, key) for name, kind, key in ROWS} for v in R})
shown = table.copy()
shown["v7 − v5"] = shown["v7"] - shown["v5"]
print("fold 평균 macro F1"); print(shown.round(3).fillna("실행 중").to_string())

large = {v: [R[v]["singles"][k]["fold_mean_macro_f1"] for k in ("ftL42", "ftL7", "ftL13") if k in R[v]["singles"]] for v in R}
print(f"\nroberta-large seed 범위 (v5): {min(large['v5']):.3f}~{max(large['v5']):.3f}  ← seed 하나로 0.01 차이는 말할 수 없다")
gap = {v: table.loc["roberta-large s42", v] - table.loc["TF-IDF (word+char)", v] for v in R}
print(f"인코더 − TF-IDF: v5 {gap['v5']:+.3f}, v7 {gap['v7']:+.3f}  ← 라벨을 실무 판단에 맞출수록(v7) 인코더가 덜 잃는다")

fig, ax = plt.subplots(figsize=(8, 3.2))
plot = table.dropna(how="all")
plot.plot.barh(ax=ax, color={"v5": "#2B5DA8", "v7": "#9AAFD1"})
ax.set_xlim(0.55, 0.72); ax.invert_yaxis(); ax.set_xlabel("fold 평균 macro F1"); ax.grid(axis="x", alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3, fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
"""한계 — 오답은 어디에 몰리고, 그 자리는 왜 안 움직이나.

두 숫자만 본다. (1) 앙상블이 오답을 줄이는 동안 경계 혼동(견적↔계약)은 거의 그대로다.
(2) 경계 혼동 조항과 뜻이 비슷한 조항은 다른 문서에서 대부분 **다른** 라벨을 받았다.
"""
S, E = R["v5"]["singles"], R["v5"]["ensembles"]
err = pd.DataFrame({
    "오답": {"TF-IDF": S["word+char TF-IDF"]["errors"], "roberta-large s42": S["ftL42"]["errors"], "앙상블": E["wc+ftB7+ftL42"]["errors"]},
    "그중 경계 혼동": {"TF-IDF": S["word+char TF-IDF"]["boundary_errors"], "roberta-large s42": S["ftL42"]["boundary_errors"], "앙상블": E["wc+ftB7+ftL42"]["boundary_errors"]},
})
err["경계 비중"] = (err["그중 경계 혼동"] / err["오답"] * 100).round(0).astype(int).astype(str) + "%"
print("v5 오답 구조 (평가 1,345건)"); print(err.to_string())

diag_path = ROOT / "reports/current/v5/cluster_diagnostics.json"
if diag_path.exists():
    cons = json.loads(diag_path.read_text(encoding="utf-8"))["cross_document_label_consistency"]
    cons_df = pd.DataFrame({g: {"건수": cons[g]["n"], "다른 문서 이웃 5건과 라벨 일치": f"{cons[g]['rate']*100:.0f}%"}
                            for g in ("모델이 맞힌 건", "경계 밖 오답", "경계 혼동")}).T
    print("\n같은 뜻 조항이 다른 RFP에서 같은 라벨을 받는 비율 (E5 이웃, 노트북 21)"); print(cons_df.to_string())
else:
    from scripts.evaluation import cluster_diagnostics as D
    import numpy as np
    rs = rows["v5"]; oof = D.load_oof("v5")
    sim = D.tfidf_similarity(rs)
    doc = np.array([r["document_id"] for r in rs]); lab = np.array([r["primary_action"] for r in rs])
    _, _, _, agree = D.cross_document_neighbors(sim, doc, lab)
    D.print_consistency(D.consistency_table(agree, D.group_masks(rs, oof, {})))
print("\n→ 정답 조항은 다른 문서에서도 같은 라벨(79%)이지만 경계 혼동 조항은 4건 중 1건만 같다.")
print("   같은 문장이 다른 곳에서는 반대로 라벨돼 있으니, 어떤 텍스트 모델이든 여기서는 틀린다. 문서 밖 정보(발주기관 사정, 제안사 입장)가 정하는 자리다.")


## 지금 시점의 결론

- **점수**: v5 앙상블 fold 평균 **0.671**(seed 범위 0.666~0.683). TF-IDF보다 +0.031, 측정 하한을 넘는 유일한 개선.
- **왜 여기서 멈추나**: 오답 401건 중 127건이 견적↔계약 경계이고, 그 자리는 비슷한 문장이 다른 RFP에서 반대 라벨을 받는 곳이다.
  텍스트 모델의 한계가 아니라 **라벨이 문서 밖 정보에 달린** 자리다. 실무자 판정 27건과 대조해도 라벨 쪽이 틀린 경우가 절반 가까이 된다.
- **라벨을 고치면**: 규칙을 실무 판단에 맞게 좁힌 v7은 사람 판정과 더 잘 맞지만(18/26 대 11/26) macro F1은 내려간다(0.640 → 0.604).
  소수 클래스가 얇아지면 이 지표는 손해다. 다만 인코더는 TF-IDF보다 덜 잃는다(격차 +0.016 → +0.036).

## 해봤고 안 된 것

| 시도 | 결과 | 어디에 |
|---|---|---|
| 입력 마스킹(양식 신호 제거) | v5에서 -0.004, v4의 +0.018은 잡음 | 노트북 14 |
| XGBoost·트리 계열 | 0.595, 앙상블에 넣어도 안 오름 | `tree_family_results.md` |
| 임베딩 군집 특징·HDBSCAN | -0.005~-0.018, 밀도 구조 없음 | 노트북 21, decisions-09 |
| 보조 헤드(blocker·원가 축 동시 학습) | 평균 -0.009, 부호가 오감 | decisions-09 09-07 09:30 |
| 문서 맥락 카드 주고 LLM 재판정 | 8:3, p=0.23, 경계는 3:3 | decisions-09 09-08 01:30 |
| 안정 라벨만 학습 | -0.087, 소수 클래스가 골라서 지워짐 | decisions-08 |
| seed 전부 투표(7명) | +0.006, 하한 아래 | decisions-09 09-08 |

## 남은 것

- **문서 확충**이 유일하게 곡선이 맞은 지렛대다. `macroF1 = 0.0896·ln(문서 수) + 0.424`(R² 0.97), +3문서에 +0.021.
- **sLLM(Qwen2.5-7B LoRA)**: 라벨 정의를 프롬프트로 읽고 세 라벨의 로그확률을 비교하는 방식. 위 점수 표에 결과가 붙는다.
  인코더와 같으면 "78억 파라미터도 경계를 못 뚫는다"가 되고, 오르면 "상식이 있는 모델이 경계 일부를 푼다"가 된다. 어느 쪽이든 결론이다.
